<div class="alert alert-warning">

# PS 10 — Reinforcement Learning in a Maze

In this problem set, you will implement and compare two approaches to sequential decision-making: **model-free reinforcement learning** (SARSA) and **model-based planning**.

## The setting

An agent must navigate a two-step maze, making a left/right choice at each step. It receives rewards at the end of each path — but the highest reward is not always on the most intuitive route. The agent must learn, through trial and error, which sequence of choices maximizes total reward.

## Model-free vs. model-based

- **SARSA** (model-free): the agent has no map of the maze. It learns purely from experience, gradually updating estimates of how good each choice is at each location.
- **Planning** (model-based): the agent has access to the full reward and transition structure. It mentally simulates all possible paths and picks the best one.

We will see that model-based planning is more accurate but more computationally expensive, while model-free learning is noisier but flexible.

We will:
1. Implement a **softmax** action-selection function
2. Implement and test the **SARSA** update rule and visualise the learned policy
3. Analyse **learning performance** across many simulations
4. Explore how **parameters** ($\beta$, $\alpha$, $\gamma$) shape learning
5. Implement **planning** and compare it to SARSA
6. Examine SARSA's robustness to **stochastic transitions**

</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

---

# The Maze

The agent makes two successive left/right choices (action 0 = left, action 1 = right). The first choice takes it from state $s=0$ to state $s=1$ (left) or $s=2$ (right). The second choice leads to one of four terminal states (3–6), as shown below.

![](Maze_JDS.png)

Each first step costs 1 point ($r = -1$). The terminal rewards differ by branch:
- Left–left (0→1→3): $-1 + 4 = 3$
- Left–right (0→1→4): $-1 + (-1) = -2$
- Right–left (0→2→5): $-1 + (-1) = -2$
- Right–right (0→2→6): $-1 + 10 = \mathbf{9}$ ← optimal

The goal is to learn the **right–right** path for a total cumulated reward of 9.

`R[s, a]` is the reward for action `a` in state `s`; `T[s, a]` is the resulting next state.

In [ ]:
R = np.array([[-1, -1],
               [ 4, -1],
               [-1, 10]])

T = np.array([[1, 2],
              [3, 4],
              [5, 6]])

# Sanity check: right (a=1) then right (a=1) -> -1 + 10 = 9
print("Optimal cumulated reward:", R[0, 1] + R[T[0, 1], 1])

---

# 1. Softmax Action Selection

The **softmax** function converts Q-values into action probabilities, with temperature parameter $\beta$ controlling the trade-off between exploration and exploitation:
$$P(a \mid s) = \frac{\exp(\beta\, Q(s,a))}{\sum_i \exp(\beta\, Q(s,a_i))}$$

To avoid numerical overflow, this can be rewritten as:
$$P(a \mid s) = \frac{1}{\sum_i \exp\!\bigl(\beta\,[Q(s,a_i) - Q(s,a)]\bigr)}$$

High $\beta$ → near-greedy (exploits best known action); $\beta = 0$ → uniform random (pure exploration).

<div class="alert alert-success">

## Exercise 1

**A.** Implement `softmax(beta, Qs)`. It should **sample and return a single action index** (integer) drawn according to the softmax probabilities over the values in `Qs`. Verify your implementation passes the tests below.

**B.** Read through `left_right_bias` — a non-learning agent that picks actions based on fixed Q-values. Run the provided code and observe the printed average rewards.

</div>

In [ ]:
def softmax(beta, Qs):
    """
    Sample an action from the softmax distribution over Q-values.

    Parameters
    ----------
    beta : float
        Inverse temperature. beta=0: uniform random; large beta: near-greedy.
    Qs : ndarray, shape (n,)
        Q-values for each of the n available actions.

    Returns
    -------
    a : int
        Sampled action index in {0, ..., n-1}.
    """
    # (TODO) replace 'pass' to compute the softmax probabilities
    pass

In [ ]:
"""Check softmax computes the correct values."""
from numpy.testing import assert_allclose

assert_allclose(softmax(100, np.array([0, 1])),       1.0)
assert_allclose(softmax(100, np.array([0, 1, 0])),    1.0)
assert_allclose(softmax(100, np.array([0, 0, 1])),    2.0)
assert_allclose(softmax(100, np.array([1, 0, 0, 0])), 0.0)

print("Success!")

In [ ]:
# (optional) add your own tests

In [ ]:
def left_right_bias(beta, Qs, n_trials):
    """
    Simulate a non-learning agent that selects actions via softmax on fixed Q-values.

    Parameters
    ----------
    beta : float
        Softmax inverse temperature.
    Qs : ndarray, shape (2,)
        Fixed Q-values for left (0) and right (1) actions.
    n_trials : int
        Number of trials to simulate.

    Returns
    -------
    float
        Mean cumulated two-step reward across all trials.
    """
    cum_r = np.zeros(n_trials)
    for i in range(n_trials):
        s0 = 0
        a0 = softmax(beta, Qs)
        r0 = R[s0, a0]
        s1 = T[s0, a0]
        a1 = softmax(beta, Qs)
        r1 = R[s1, a1]
        cum_r[i] = r0 + r1
    return np.mean(cum_r)


n_trials = 10_000

beta, Qs = 0, np.zeros(2)
print(f"Random choice:               {left_right_bias(beta, Qs, n_trials):.2f}")

beta, Qs = 5, np.array([0.8, 0.5])
print(f"Left-biased  (beta=5):       {left_right_bias(beta, Qs, n_trials):.2f}")

beta, Qs = 5, np.array([0.5, 0.8])
print(f"Right-biased (beta=5):       {left_right_bias(beta, Qs, n_trials):.2f}")

beta, Qs = 50, np.array([0.5, 0.8])
print(f"Near-greedy right (beta=50): {left_right_bias(beta, Qs, n_trials):.2f}")

---

# 2. SARSA

**SARSA** (State–Action–Reward–State–Action) is an on-policy TD learning algorithm. After each transition $(s_t, a_t, r_t, s_{t+1}, a_{t+1})$ it updates the Q-value estimate:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha\bigl[r_t + \gamma\, Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)\bigr]$$

At terminal states (end of the maze) there is no next action, so:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha\bigl[r_t - Q(s_t, a_t)\bigr]$$

<div class="alert alert-success">

## Exercise 2A

**A. SARSA update.** Implement `sarsa(alpha, gamma, Q, s, a, r)`. It receives a complete two-step maze path and returns the updated Q-table. Verify your implementation passes the tests.


</div>

In [ ]:
def sarsa(alpha, gamma, Q, s, a, r):
    """
    Apply one SARSA update for a two-step maze path.

    Parameters
    ----------
    alpha : float
        Learning rate.
    gamma : float
        Discount factor.
    Q : ndarray, shape (3, 2)
        Current Q-values for states {0, 1, 2} and actions {0, 1}.
    s : ndarray of int, shape (2,)
        Two successive states visited: [s_t, s_{t+1}].
    a : ndarray of int, shape (2,)
        Two successive actions taken: [a_t, a_{t+1}].
    r : ndarray of float, shape (2,)
        Two successive rewards received: [r_t, r_{t+1}].

    Returns
    -------
    Q : ndarray, shape (3, 2)
        Updated Q-table (copy — original is not modified).
    """
    Q = Q.copy()
    # (TODO) First step: bootstrap off Q[s1, a1] before it is updated

    # (TODO) Second step: terminal — no next state

    return Q

In [ ]:
"""Check sarsa computes the correct values."""
from numpy.testing import assert_allclose

s = np.array([0, 1]).astype(int)
a = np.zeros(2).astype(int)
r = np.array([-1, 4]).astype(int)
alpha, gamma = 0.1, 0.9

assert_allclose(sarsa(alpha, gamma, 0.5 * np.ones([3, 2]), s, a, r),
                np.array([[0.395, 0.5], [0.85, 0.5], [0.5, 0.5]]))
a = np.ones(2).astype(int)
assert_allclose(sarsa(alpha, gamma, 0.5 * np.ones([3, 2]), s, a, r),
                np.array([[0.5, 0.395], [0.5, 0.85], [0.5, 0.5]]))
s = np.array([0, 2]).astype(int)
assert_allclose(sarsa(alpha, gamma, 0.5 * np.ones([3, 2]), s, a, r),
                np.array([[0.5, 0.395], [0.5, 0.5], [0.5, 0.85]]))
alpha = 0.2
assert_allclose(sarsa(alpha, gamma, 0.5 * np.ones([3, 2]), s, a, r),
                np.array([[0.5, 0.29], [0.5, 0.5], [0.5, 1.2]]))
Q = sarsa(alpha, gamma, 0.5 * np.ones([3, 2]), s, a, r)
gamma = 0.75
assert_allclose(sarsa(alpha, gamma, Q, s, a, r),
                np.array([[0.5, 0.212], [0.5, 0.5], [0.5, 1.76]]))

print("Success!")

In [ ]:
# (optional) add your own test cases

<div class="alert alert-success">

## Exercise 2B

**B. One-trial SARSA.** Implement `onetrial_sarsa(parameters, Q, R, T)`, which navigates one path through the maze using softmax and calls `sarsa` to update Q. Use `left_right_bias` as a structural guide. Return the updated Q-table and the sequences of actions and rewards.


</div>

In [ ]:
def onetrial_sarsa(parameters, Q, R, T):
    """
    Run one trial of SARSA: navigate the maze and update Q-values.

    Parameters
    ----------
    parameters : ndarray, shape (3,)
        [beta, alpha, gamma].
    Q : ndarray, shape (3, 2)
        Current Q-value table.
    R : ndarray, shape (3, 2)
        Reward function.
    T : ndarray, shape (3, 2)
        Transition function.

    Returns
    -------
    Q : ndarray, shape (3, 2) — updated Q-table
    a : ndarray, shape (2,)  — actions taken
    r : ndarray, shape (2,)  — rewards received
    """
    beta, alpha, gamma = parameters
    s = np.zeros(2, dtype=int)
    a = np.zeros(2, dtype=int)
    r = np.zeros(2)
    # (TODO): replace 'YOUR_CODE_HERE' to complete the code
    s[0]    = 'YOUR_CODE_HERE'
    a[0]    = 'YOUR_CODE_HERE'
    r[0]    = 'YOUR_CODE_HERE'
    s[1]    = 'YOUR_CODE_HERE'
    a[1]    = 'YOUR_CODE_HERE'
    r[1]    = 'YOUR_CODE_HERE'

    Q = 'YOUR_CODE_HERE'
    return Q, a, r

In [ ]:
# (optional) add your own test cases

<div class="alert alert-success">

## Exercise 2C&D

**C.** Run the provided code to plot how Q-values evolve over 100 trials. Run it a few times and upload the most representative figure to Gradescope.

**D.** Use the Q-values from Part C to visualise the **learned policy**: plot the final Q-values for each state as a grouped bar chart (left vs. right) and annotate which action the agent prefers at each state.

</div>

In [ ]:
# Exercise 2C: Q-value evolution over 100 trials
n_trials = 100
Q = np.full((3, 2), 0.5)
beta, alpha, gamma = 5, 0.1, 1
parameters = np.array([beta, alpha, gamma])

Qs = np.empty((6, n_trials))
for t in range(n_trials):
    # (TODO): replace 'pass' to update Q-values
    pass

fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Q(s0,a0)', 'Q(s0,a1)', 'Q(s1,a0)', 'Q(s1,a1)', 'Q(s2,a0)', 'Q(s2,a1)']
for i, label in enumerate(labels):
    # (TODO): replace 'pass' to plot Q-values
    pass

ax.set_xlabel('Trial')
ax.set_ylabel('Q-value')
ax.set_title('SARSA: Q-value evolution over 100 trials')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 2D: visualise the learned policy as a grouped bar chart
# Q holds the final Q-values from the 100-trial run above
state_labels = ['State 0\n(start)', 'State 1\n(left branch)', 'State 2\n(right branch)']
x = np.arange(3)
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
# (TODO): plot the final Q-values for each state as a grouped bar chart (left vs. right); hint: use `ax.bar` (two lines)

# (TODO): reaplce 'pass' to annotate preferred action at each state
for i in range(3):
    pass

ax.set_xticks(x)
ax.set_xticklabels(state_labels)
ax.set_ylabel('Q-value')
ax.set_title('Learned Q-values and preferred actions after 100 trials')
ax.legend()
plt.tight_layout()
plt.show()

---

# 3. Performance Across Simulations

Running SARSA once is noisy. We need many simulations to understand what the algorithm typically learns.

<div class="alert alert-success">

## Exercise 3A

**A.** Run 100 independent simulations of 50 SARSA trials each ($\beta=5$, $\alpha=0.1$, $\gamma=1$). Re-initialise Q to 0.5 for each simulation. Store the final Q-values and create three scatter plots — one per decision state — showing Q(s,a0) vs Q(s,a1):

- subplot a: State 0 — first-step policy
- subplot b: State 1 — second-step policy from the left branch
- subplot c: State 2 — second-step policy from the right branch


</div>

In [ ]:
# Exercise 3A: 100 simulations — scatter final Q-values
n_trials   = 50
n_sims     = 100
beta, alpha, gamma = 5, 0.1, 1
parameters = np.array([beta, alpha, gamma])

Q_final = np.empty((n_sims, 3, 2))
for sim in range(n_sims):
    # (TODO): replace 'pass' to run simulation
    pass

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
state_names = ['State 0 (first choice)',
               'State 1 (left branch)',
               'State 2 (right branch)']
for i, (ax, name) in enumerate(zip(axes, state_names)):
    # (TODO): replace 'pass' to plot different state Q values
    pass

plt.suptitle('Final Q-values across 100 simulations', y=1.02)
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3B

**B.** Plot the **mean** final Q-values from Part A as a grouped bar chart (same format as Exercise 2D). Does the average learned policy match what you expect?

</div>

In [ ]:
# Exercise 3B: mean final Q-values — average learned policy
Q_mean = Q_final.mean(axis=0)   # shape (3, 2)

state_labels = ['State 0\n(start)', 'State 1\n(left branch)', 'State 2\n(right branch)']
x = np.arange(3)
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
# (TODO): plot the mean final Q-values for each state as a grouped bar chart (left vs. right); hint: use `ax.bar` (two lines)

# (TODO): reaplce 'pass' to annotate preferred action at each state
for i in range(3):
   pass

ax.set_xticks(x)
ax.set_xticklabels(state_labels)
ax.set_ylabel('Mean Q-value (across 100 simulations)')
ax.set_title('Average learned policy across 100 simulations')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3C

**C.** In your scatter plots from Part A you should see a small cluster of simulations with clearly different Q-values from the main cluster, consistently across all three states. In 2–3 sentences: what happened in those simulations?

</div>

**3C. Your Answer:** 

---

# 4. Parameter Effects

To compare settings, we summarise performance as the **average cumulated two-step reward per trial** across simulations (ranging from $-2$ at worst to $9$ at best).

<div class="alert alert-success">

## Exercise 4A

**Setup.** Implement `cumulative_outcome(parameters, R, T, n_trials, n_sims)`, which runs `n_sims` independent SARSA simulations of `n_trials` trials each and returns the mean cumulated reward at every trial (shape `(n_trials,)`).

**A.** Effect of $\beta$. Using $\gamma = 1$, $n_\text{trials} = 20$, $n_\text{sims} = 1000$: plot the **final-trial** average reward as a function of $\beta \in \{1, 3, 5, \ldots, 19\}$, for $\alpha = 0.1$ and $\alpha = 0.3$ on the same figure.


</div>

In [ ]:
def cumulative_outcome(parameters, R, T, n_trials, n_sims):
    """
    Estimate average cumulated two-step reward per trial across simulations.

    Parameters
    ----------
    parameters : ndarray, shape (3,)
        [beta, alpha, gamma].
    R : ndarray, shape (3, 2) — reward function
    T : ndarray, shape (3, 2) — transition function
    n_trials : int — trials per simulation
    n_sims   : int — number of independent simulations

    Returns
    -------
    ndarray, shape (n_trials,)
        Mean cumulated two-step reward at each trial, averaged over simulations.
    """
    rewards = np.zeros((n_sims, n_trials))
    for sim in range(n_sims):
        # (TODO): replace 'pass' to complete the code
        pass
    
    return rewards.mean(axis=0)

In [ ]:
# Exercise 4A: effect of beta on final performance
n_trials, n_sims, gamma = 20, 1000, 1
beta_range = np.arange(1, 20, 2)

fig, ax = plt.subplots(figsize=(7, 4))
for alpha, color in zip([0.1, 0.3], ['steelblue', 'tomato']):
    # (TODO): replace 'pass' to complete the code
    pass


ax.set_xlabel(r'$\beta$ (inverse temperature)')
ax.set_ylabel('Mean final cumulated reward')
ax.set_title(r'Effect of $\beta$ on SARSA performance')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4B

**B.** In 2–3 sentences: how does $\beta$ affect performance? Are there differences between the two $\alpha$ values, and why?


</div>

**4B. Your Answer:**

<div class="alert alert-success">

## Exercise 4C
**C.** Effect of $\alpha$. Same setup: plot final-trial reward vs $\alpha \in \{0.05, 0.10, \ldots, 0.95\}$, for $\beta = 1$ and $\beta = 5$ on the same figure.

</div>

In [ ]:
# Exercise 4C: effect of alpha on final performance
n_trials, n_sims, gamma = 20, 1000, 1
alpha_range = np.arange(0.05, 1.0, 0.05)

fig, ax = plt.subplots(figsize=(7, 4))
for beta, color in zip([1, 5], ['steelblue', 'tomato']):
    # (TODO): replace 'pass' to complete the code
    pass

ax.set_xlabel(r'$\alpha$ (learning rate)')
ax.set_ylabel('Mean final cumulated reward')
ax.set_title(r'Effect of $\alpha$ on SARSA performance')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4D
**D.** In 2–3 sentences: how does $\alpha$ affect performance, and does the effect depend on $\beta$?

</div>

**4D. Your Answer:** 

<div class="alert alert-success">

## Exercise 4E
**E.** Effect of $\gamma$. Using $\beta = 5$, $\alpha = 0.1$, $n_\text{trials} = 20$, $n_\text{sims} = 1000$: plot the **full learning curve** (all 20 trials) for $\gamma \in \{0.0, 0.25, 0.5, 0.75, 1.0\}$ on a single figure. Add a legend.
</div>

In [ ]:
# Exercise 4E: effect of gamma on the full learning curve
n_trials, n_sims, beta, alpha = 20, 1000, 5, 0.1
gamma_vals  = [0.0, 0.25, 0.5, 0.75, 1.0]
colors      = plt.cm.viridis(np.linspace(0.1, 0.9, len(gamma_vals)))

fig, ax = plt.subplots(figsize=(8, 4))
for gamma, color in zip(gamma_vals, colors):
    # (TODO): replace 'pass' to complete the code
    pass

ax.set_xlabel('Trial')
ax.set_ylabel('Mean cumulated reward')
ax.set_title(r'Effect of $\gamma$ on SARSA learning curve')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4F

**F.** In 2–3 sentences: why does low $\gamma$ hurt performance in this particular maze? What special role does $\gamma = 0$ play?

</div>

**4F. Your Answer:** 

---

# 5. Planning

In **model-based planning**, the agent has direct access to `R` and `T`. Rather than learning from experience, it mentally simulates all four paths, evaluates the discounted cumulated reward for each, and selects a path via softmax.

<div class="alert alert-success">

## Exercise 5A

**A.** Implement `onetrial_planning(parameters, R, T)`. Your function should:
1. Compute the cumulated discounted reward $Q_\text{path}$ for each of the four paths (left–left, left–right, right–left, right–right).
2. Select a path using `softmax` over these four values.
3. Return `Q_path` (shape `(4,)`), actions `a` (shape `(2,)`), and rewards `r` (shape `(2,)`).

Verify your implementation passes the tests. Then run 1000 trials with $\beta = 0.5$, $\gamma = 0.5$ and plot the proportion of times each path was chosen as a bar chart.


</div>

In [ ]:
def onetrial_planning(parameters, R, T):
    """
    Run one trial of model-based planning.

    Parameters
    ----------
    parameters : ndarray, shape (2,)
        [beta, gamma].
    R : ndarray, shape (3, 2) — reward function
    T : ndarray, shape (3, 2) — transition function

    Returns
    -------
    Q_path : ndarray, shape (4,)
        Cumulated discounted reward for each path:
        index = a0*2 + a1 → [LL, LR, RL, RR].
    a : ndarray, shape (2,) — actions on chosen path
    r : ndarray, shape (2,) — rewards on chosen path
    """
    beta, gamma = parameters

    # (TODO) replace pass to evaluate all four paths: a0 in {0,1}, a1 in {0,1}
    pass

In [ ]:
"""Check onetrial_planning computes the correct path values."""
from numpy.testing import assert_allclose

assert_allclose(
    onetrial_planning(np.array([1, 1]),   np.array([[-1,-1],[4,-1],[-1,10]]), T)[0],
    np.array([3, -2, -2, 9]))
assert_allclose(
    onetrial_planning(np.array([1, 0.5]), np.array([[-1,-1],[4,-1],[-1,10]]), T)[0],
    np.array([1, -1.5, -1.5, 4]))

print("Success!")

In [ ]:
# (Optional)Add your own test cases here

In [ ]:
# Exercise 5A: run 1000 planning trials and plot path frequencies
n_runs = 1000
path_counts = np.zeros(4)

for _ in range(n_runs):
    # (TODO): replace 'pass' to complete the code
    pass

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Left-Left', 'Left-Right', 'Right-Left', 'Right-Right'],
       path_counts / n_runs, color='steelblue', edgecolor='k')
ax.set_ylabel('Proportion of trials')
ax.set_title(r'Planning path frequencies ($\beta=0.5$, $\gamma=0.5$)')
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 5B

**B.** Run both SARSA ($\beta=5$, $\alpha=0.1$, $\gamma=1$) and Planning ($\beta=5$, $\gamma=1$) for 500 simulations of 100 trials each. Plot their mean reward learning curves on the same axes. What difference do you observe, and why?

</div>

In [ ]:
# Exercise 5B: compare SARSA vs Planning reward learning curves
n_trials, n_sims = 50, 500
params_sarsa     = np.array([1, 0.5, 1.0])
params_planning  = np.array([1, 1.0])

sarsa_rew = np.zeros((n_sims, n_trials))
plan_rew  = np.zeros((n_sims, n_trials))

for sim in range(n_sims):
    Q = np.full((3, 2), 0.5)
    for t in range(n_trials):
        # (TODO): replace 'pass' to complete the code
        pass

for sim in range(n_sims):
    for t in range(n_trials):
        # (TODO): replace 'pass' to complete the code
        pass

fig, ax = plt.subplots(figsize=(7, 4))
# (TODO): Plot sarsa_rew and plan_rew mean reward learning curves on the same axes. Hint: use `ax.plot` (two lines)

ax.set_xlabel('Trial')
ax.set_ylabel('Mean cumulated reward')
ax.set_title('SARSA vs Planning: reward learning curves')
ax.legend()
plt.tight_layout()
plt.show()

**5B. Your Answer:** 

---

# 6. Adapting to Environmental Change

So far, both agents operated in a **static** environment. What happens when the world changes?

At trial 50, the maze is reconfigured: going **right** at S0 now leads to S1 (the low-reward branch) with probability **0.8**, and going **left** leads to S2 (the high-reward branch) with probability **0.8**. The optimal strategy has flipped.

The Planning agent is **not** informed of this change — it continues using the original transition model. SARSA has no model to update; it just keeps learning from whatever it experiences.

<div class="alert alert-success">

## Exercise 6A

**A.** Implement two functions for the changed environment:

- `onetrial_sarsa_envchange(parameters, Q, R, T)`: same as `onetrial_sarsa`, except the transition at S0 is stochastic — the chosen action leads to the *opposite* branch with probability 0.8. SARSA observes the actual outcome and updates Q as usual.
- `onetrial_planning_envchange(parameters, R, T)`: the agent plans a full path using the original (now wrong) `T`, commits to both actions, then executes them in the actual stochastic environment. No Q-table is updated.


</div>

In [ ]:
def onetrial_sarsa_envchange(parameters, Q, R, T):
    beta, alpha, gamma = parameters
    s = np.zeros(2, dtype=int)
    a = np.zeros(2, dtype=int)
    r = np.zeros(2)

    # (TODO) replace 'YOUR_CODE_HERE' to complete the code
    # Start from initial state 0
    s[0] = 'YOUR_CODE_HERE'

    # Choose first action from Q(s0, :)
    a[0] = 'YOUR_CODE_HERE'

    # Observe immediate reward for (s0, a0)
    r[0] = 'YOUR_CODE_HERE'

    # Transition to second state with environment change/noise:
    # 80%: go to the opposite branch of chosen action
    # 20%: follow the originally selected branch
    s[1] = 'YOUR_CODE_HERE'

    # Choose second action from policy in new state
    a[1] = 'YOUR_CODE_HERE'

    # Observe second-step reward
    r[1] = 'YOUR_CODE_HERE'

    # Perform one SARSA update using trajectory (s, a, r)
    Q = 'YOUR_CODE_HERE'   

    # Return updated value table and trial behavior
    return Q, a, r


def onetrial_planning_envchange(parameters, R, T):
    beta, gamma = parameters
    # (TODO) replace 'YOUR_CODE_HERE' to complete the code
    # Compute model-based values for all 2-step action paths (a0, a1)
    # using the assumed transition model T:
    # Q_path[a0,a1] = immediate reward at step 1 + discounted reward at step 2
    Q_path = 'YOUR_CODE_HERE'

    # Select one of the 4 candidate paths via softmax over planned path values
    path_idx = softmax(beta, Q_path)
    a0, a1 = path_idx // 2, path_idx % 2

    # Execute first action in actual environment with changed dynamics:
    # 80%: transition to opposite branch from intended a0
    # 20%: transition as planned
    s1_actual = 'YOUR_CODE_HERE'
    # Return realized rewards from executed trajectory
    # (note: second reward uses actual reached state and planned second action a1)    
    return np.array([R[0, a0], R[s1_actual, a1]])


<div class="alert alert-success">

## Exercise 6B

**B.** Run 500 simulations of 100 trials. Use the original trial functions for trials 1–50, then switch to the `_envchange` variants for trials 51–100 (SARSA carries its Q-table across the change). Plot the mean reward learning curves for both agents, with a vertical dashed line at the change point.


</div>

In [ ]:
# Exercise 6B: learning curves across the environment change
n_pre, n_post    = 50, 100
n_trials, n_sims = n_pre + n_post, 500
params_sarsa     = np.array([1, 0.5, 1.0])
params_planning  = np.array([1, 1.0])

sarsa_rew = np.zeros((n_sims, n_trials))
# (TODO) replace 'pass' to complete the code
for sim in range(n_sims):
    Q = np.full((3, 2), 0.5)
    for t in range(n_pre):
        pass
    
    for t in range(n_pre, n_trials):
        pass

plan_rew  = np.zeros((n_sims, n_trials))
for sim in range(n_sims):
    for t in range(n_pre):
        pass
    
    for t in range(n_pre, n_trials):
        pass

fig, ax = plt.subplots(figsize=(8, 4))
# (TODO): Plot sarsa_rew and plan_rew mean reward learning curves on the same axes. Hint: use `ax.plot` (two lines)
ax.axvline(n_pre - 0.5, color='k', linestyle='--', linewidth=1.5, label='Environment changes')
ax.set_xlabel('Trial')
ax.set_ylabel('Mean cumulated reward')
ax.set_title('Adapting to an environmental change')
ax.legend()
plt.tight_layout()
plt.show()


<div class="alert alert-success">

## Exercise 6C
**C.** In 2–3 sentences, explain what you observe and why.

</div>

**6C. Your Answer:** 

---

<div class="alert alert-warning">

# When you are done:

**Restart the kernel and re-run the whole notebook** (`Kernel → Restart & Run All`) to make sure everything runs without errors.

</div>